# Orbital chaos from exported engine runs

A PROTOTYPE of the compute gate (Prompt 268, COMPUTE_GATE_II.md): it is not in the catalogue and not shipped. It reads its input as the text `gravitas_input` and its settings as the JSON text `gravitas_params`, and returns `gravitas_output`, a dict of numbers. Python only, numpy and scipy only.


In [ ]:
import base64, json, math
import numpy as np
doc = json.loads(gravitas_input)
assert doc.get("format") == "gravitas.grid" and doc.get("formatVersion") == 1, "this notebook reads gravitas.grid/1"
assert [a["name"] for a in doc["axes"]] == ["snapshot"] and [o["name"] for o in doc["outputs"]] == ["separation"], "the grid must have the axis snapshot and the output separation"
n = len(doc["axes"][0]["values"])
raw = np.frombuffer(base64.b64decode(doc["storage"]["data"]), dtype="<f8")
assert len(raw) == n and n >= 40 and bool(np.all(np.isfinite(raw))) and bool(np.all(raw > 0)), "the separation must be a positive finite series of at least 40 snapshots"
sep = raw.astype(float)
t = 0.1 * np.arange(n)

In [ ]:
S = float(np.median(sep[-20:]))
s0 = float(sep[0])
grew = np.nonzero(sep > 3 * s0)[0]
if len(grew) == 0:
    raise ValueError("not constrained: the separation never exceeds three times its start")
i0 = int(grew[0])
over = np.nonzero((sep > 0.1 * S) & (np.arange(n) > i0))[0]
i1 = int(over[0]) - 1 if len(over) else n - 1
if i1 - i0 + 1 < 10:
    raise ValueError("not constrained: the growth window is %d snapshots long (i0=%r i1=%r), fewer than 10" % (i1 - i0 + 1, i0, i1))
tt, yy = t[i0:i1 + 1], np.log(sep[i0:i1 + 1])
tm, ym = tt.mean(), yy.mean()
slope = float(((tt - tm) * (yy - ym)).sum() / ((tt - tm) ** 2).sum())
res = yy - (ym + slope * (tt - tm))
r2 = 1.0 - float((res ** 2).sum()) / float(((yy - ym) ** 2).sum())
if r2 < 0.98:
    raise ValueError("not constrained: ln(separation) is not a straight line in time over the window (R2=%r i0=%r i1=%r)" % (r2, i0, i1))
gravitas_output = {"answer": {"lambda": slope, "lyapunovTime": 1.0 / slope, "r2": r2, "i0": i0, "i1": i1, "S": S}}